In [55]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset
import numpy as np
import pandas as pd

In [56]:
class CrudeDataset(Dataset):
    """
    Multi-channel temporal dataset for spoofing detection.

    Output:
        features: (seq_len, num_channels, feature_dim)
        target:   (num_channels, target_dim)
    """

    mapping = {f"ch{i}": i for i in range(8)}

    def __init__(self, paths, sequence_length=50):
        self.sequence_length = sequence_length

        dfs = []
        targets = []

        # =========================
        # 1. LOAD + PREPROCESS
        # =========================
        for path in paths:
            df = pd.read_csv(path)

            # Sort by time
            df = df.sort_values("time")

            # Separate target
            tgt = df[["time", "spoofed"]].copy()

            # Drop target from features
            df = df.drop(columns=["spoofed"])

            # Encode channel
            df["channel"] = df["channel"].map(self.mapping)

            # Set index
            df = df.set_index("time")
            tgt = tgt.set_index("time")

            dfs.append(df)
            targets.append(tgt)

        # =========================
        # 2. TIME ALIGNMENT (CRITICAL)
        # =========================
        common_index = dfs[0].index

        for df in dfs[1:]:
            common_index = common_index.intersection(df.index)

        # Apply alignment
        self.df = [df.loc[common_index].sort_index() for df in dfs]
        self.targets = [t.loc[common_index].sort_index() for t in targets]

        # Dataset length
        self.length = len(common_index)

        # =========================
        # 3. SANITY CHECKS
        # =========================
        assert self.length > sequence_length, "Sequence length too large!"

        # Feature dimension consistency
        feature_dims = [df.shape[1] for df in self.df]
        assert len(set(feature_dims)) == 1, "Feature dims mismatch across files!"

        self.feature_dim = feature_dims[0]
        self.num_channels = len(self.df)

    # =========================
    # 4. DATASET LENGTH
    # =========================
    def __len__(self):
        return self.length - self.sequence_length

    # =========================
    # 5. GET ITEM
    # =========================
    def __getitem__(self, idx):
        start = idx
        end = idx + self.sequence_length
        target_idx = end - 1

        features = []
        targets = []

        # Collect per-channel sequences
        for df, tgt in zip(self.df, self.targets):
            seq = df.iloc[start:end].values                # (seq_len, feature_dim)
            target_val = tgt.iloc[target_idx].values       # (target_dim,)

            features.append(seq)
            targets.append(target_val)

        # =========================
        # 6. SHAPE STANDARDIZATION
        # =========================
        # (num_channels, seq_len, feature_dim) → (seq_len, num_channels, feature_dim)
        # features = np.stack(features, axis=1)

        # (num_channels, target_dim)
        # targets = np.array(targets)

        # =========================
        # 7. TO TENSOR
        # =========================
        features = torch.tensor(features, dtype=torch.float32)
        targets = torch.tensor(targets, dtype=torch.float32)

        # =========================
        # 8. NUMERICAL STABILITY
        # =========================
        features = torch.nan_to_num(features, nan=0.0, posinf=10.0, neginf=-10.0)
        targets = torch.nan_to_num(targets, nan=0.0, posinf=10.0, neginf=-10.0)

        features = torch.clamp(features, min=-5.0, max=5.0)
        targets = torch.clamp(targets, min=-5.0, max=5.0)

        return features, targets

In [59]:
train_base="../dataset/train/"
val_base="../dataset/val/"
train_paths=[]
val_paths=[]
for i in range(0,8):
    filename=f"ch{i}.csv"
    train_filepath=train_base+"train_"+filename
    val_filepath=val_base+"val_"+filename
    print(train_filepath)
    print(val_filepath)
    train_paths.append(train_filepath)
    val_paths.append(val_filepath)

../dataset/train/train_ch0.csv
../dataset/val/val_ch0.csv
../dataset/train/train_ch1.csv
../dataset/val/val_ch1.csv
../dataset/train/train_ch2.csv
../dataset/val/val_ch2.csv
../dataset/train/train_ch3.csv
../dataset/val/val_ch3.csv
../dataset/train/train_ch4.csv
../dataset/val/val_ch4.csv
../dataset/train/train_ch5.csv
../dataset/val/val_ch5.csv
../dataset/train/train_ch6.csv
../dataset/val/val_ch6.csv
../dataset/train/train_ch7.csv
../dataset/val/val_ch7.csv


In [60]:
train_dataset = CrudeDataset(train_paths,50)
val_dataset = CrudeDataset(val_paths,50)

In [61]:
print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")

Train: 77931 samples
Val: 33371 samples


In [62]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [63]:
feature, target = next(iter(train_loader))

print(feature.shape)
print(target.shape)

torch.Size([32, 8, 50, 14])
torch.Size([32, 8, 1])


In [65]:
from tqdm import tqdm